In [3]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

def find_root() -> Path:
    p = Path.cwd()
    while p != p.parent:
        if (p / "data").exists() and (p / "scripts").exists():
            return p
        p = p.parent
    raise RuntimeError("Root folder not found")


ROOT = find_root()

INPUT_FILE = ROOT / "data" / "processed" / "features" / "phase_features_long.csv"
OUTPUT_DIR = ROOT / "reports" / "phase_delta_plots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PHASE_ORDER = [
    "standing_1",
    "walk_low_1",
    "walk_mod_1",
    "walk_low_2",
    "walk_mod_2",
    "standing_2",
]

CONDITION_ORDER = [
    "no_wind",
    "wind_low",
    "wind_high",
]

SIGNALS_TO_PLOT = [
    "t_dorsal",
    "t_microclimateback",
    "rh_microclimateback",
    "mean_skin_temperature",
    "t_abdomen",
    "t_chest",
    "t_neck",
    "t_hand",
    "t_shin",
    "rh_dorsal",
    "rh_abdomen",
    "rh_chest",
    "rh_neck",
    "rh_hand",
    "rh_shin",
    "t_microclimatefront",
    "rh_microclimatefront",
    "t_environment",
    "rh_environment",
    "t_microclimate_gradient",
    "rh_microclimate_gradient",
]


def load_data() -> pd.DataFrame:
    if not INPUT_FILE.exists():
        raise FileNotFoundError(f"Missing file: {INPUT_FILE}")

    df = pd.read_csv(INPUT_FILE)

    for col in ["mean", "std", "median", "min", "max", "delta_in_out", "slope", "baseline_value", "delta_mean_from_baseline"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "phase" in df.columns:
        df["phase"] = pd.Categorical(df["phase"], categories=PHASE_ORDER, ordered=True)

    if "condition" in df.columns:
        df["condition"] = df["condition"].astype(str).str.strip()
        df["condition"] = pd.Categorical(df["condition"], categories=CONDITION_ORDER, ordered=True)

    if "signal" in df.columns:
        df["signal"] = df["signal"].astype(str).str.strip()

    return df


def get_available_signals(df: pd.DataFrame) -> list[str]:
    available = sorted(df["signal"].dropna().unique().tolist())
    return [s for s in SIGNALS_TO_PLOT if s in available]


def make_phase_delta_plot(df: pd.DataFrame, signal: str) -> None:
    sub = df[df["signal"] == signal].copy()
    sub = sub.dropna(subset=["condition", "phase", "delta_mean_from_baseline"])

    if sub.empty:
        print(f"Skipping {signal}: no valid rows")
        return

    plot_df = (
        sub.groupby(["condition", "phase"], observed=True)["delta_mean_from_baseline"]
        .agg(["mean", "std", "count"])
        .reset_index()
    )

    if plot_df.empty:
        print(f"Skipping {signal}: grouped dataframe empty")
        return

    plot_df["sem"] = plot_df["std"] / plot_df["count"].pow(0.5)

    plt.figure(figsize=(8, 5))

    for condition in CONDITION_ORDER:
        cond_df = plot_df[plot_df["condition"] == condition].sort_values("phase")
        if cond_df.empty:
            continue

        x = cond_df["phase"].astype(str).tolist()
        y = cond_df["mean"].to_numpy(dtype=float)
        sem = cond_df["sem"].fillna(0).to_numpy(dtype=float)

        plt.plot(x, y, marker="o", label=condition)
        plt.fill_between(range(len(x)), y - sem, y + sem, alpha=0.2)

    plt.axhline(0, linestyle="--", linewidth=1)

    plt.xlabel("Phase")
    plt.ylabel(f"Δ from baseline")
    plt.title(f"{signal}: phase response by condition")
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()

    out_path = OUTPUT_DIR / f"{signal}_delta_by_phase.png"
    plt.savefig(out_path, dpi=300)
    plt.close()

    print(f"Saved: {out_path}")


def main() -> None:
    df = load_data()
    signals = get_available_signals(df)

    if not signals:
        raise ValueError("No target signals found in phase_features_long.csv")

    print("Signals to plot:")
    for s in signals:
        print(" -", s)

    for signal in signals:
        make_phase_delta_plot(df, signal)

    print(f"\nDone. Plots saved in: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()

Signals to plot:
 - t_dorsal
 - t_microclimateback
 - rh_microclimateback
 - mean_skin_temperature
 - t_abdomen
 - t_chest
 - t_neck
 - t_hand
 - t_shin
 - rh_dorsal
 - rh_abdomen
 - rh_chest
 - rh_neck
 - rh_hand
 - rh_shin
 - t_microclimatefront
 - rh_microclimatefront
 - t_environment
 - rh_environment
Saved: /home/eleonorab/repository/convective_cooling/reports/phase_delta_plots/t_dorsal_delta_by_phase.png
Saved: /home/eleonorab/repository/convective_cooling/reports/phase_delta_plots/t_microclimateback_delta_by_phase.png
Saved: /home/eleonorab/repository/convective_cooling/reports/phase_delta_plots/rh_microclimateback_delta_by_phase.png
Saved: /home/eleonorab/repository/convective_cooling/reports/phase_delta_plots/mean_skin_temperature_delta_by_phase.png
Saved: /home/eleonorab/repository/convective_cooling/reports/phase_delta_plots/t_abdomen_delta_by_phase.png
Saved: /home/eleonorab/repository/convective_cooling/reports/phase_delta_plots/t_chest_delta_by_phase.png
Saved: /home/eleo

In [10]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


def find_root() -> Path:
    p = Path.cwd()
    while p != p.parent:
        if (p / "data").exists() and (p / "scripts").exists():
            return p
        p = p.parent
    raise RuntimeError("Root folder not found")


ROOT = find_root()
INPUT_DIR = ROOT / "data" / "processed" / "aggregated_by_signal"
OUTPUT_DIR = ROOT / "reports" / "mechanistic_plots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONDITION_ORDER = ["no_wind", "wind_low", "wind_high"]
BIN_SECONDS = 10


def load_signal(signal_file: str, value_name: str) -> pd.DataFrame:
    path = INPUT_DIR / signal_file
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    df = pd.read_csv(path)

    required_cols = ["subject_id", "condition", "elapsed_seconds", "elapsed_minutes", "value"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in {path.name}: {missing}")

    df = df[required_cols].copy()
    df["subject_id"] = df["subject_id"].astype(str).str.strip()
    df["condition"] = df["condition"].astype(str).str.strip()
    df["elapsed_seconds"] = pd.to_numeric(df["elapsed_seconds"], errors="coerce")
    df["elapsed_minutes"] = pd.to_numeric(df["elapsed_minutes"], errors="coerce")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.dropna(subset=["subject_id", "condition", "elapsed_seconds", "value"]).copy()

    df["elapsed_seconds_bin"] = (df["elapsed_seconds"] / BIN_SECONDS).round() * BIN_SECONDS
    df["elapsed_minutes_bin"] = df["elapsed_seconds_bin"] / 60.0

    df = (
        df.groupby(
            ["subject_id", "condition", "elapsed_seconds_bin", "elapsed_minutes_bin"],
            as_index=False
        )["value"]
        .mean()
    )

    df = df.rename(columns={"value": value_name})
    return df


def build_pair_df(file_x: str, name_x: str, file_y: str, name_y: str) -> pd.DataFrame:
    dx = load_signal(file_x, name_x)
    dy = load_signal(file_y, name_y)

    merged = pd.merge(
        dx,
        dy,
        on=["subject_id", "condition", "elapsed_seconds_bin", "elapsed_minutes_bin"],
        how="inner"
    )

    merged = merged.dropna(subset=[name_x, name_y]).copy()

    print(f"\nPAIR DATASET: {name_y} vs {name_x}")
    print("shape:", merged.shape)
    print("conditions:", merged["condition"].value_counts(dropna=False).to_dict())

    return merged


def print_pair_debug(df: pd.DataFrame, x: str, y: str) -> None:
    print(f"\nDEBUG for {y} vs {x}")
    for cond in CONDITION_ORDER:
        sub = df[df["condition"] == cond].copy()
        print(f"\nCONDITION: {cond}")
        print("rows:", len(sub))
        if len(sub) > 0:
            print(f"{x} non-null:", sub[x].notna().sum())
            print(f"{y} non-null:", sub[y].notna().sum())


def get_local_limits(plot_df: pd.DataFrame, x: str, y: str) -> tuple[tuple[float, float], tuple[float, float]]:
    x_low, x_high = plot_df[x].quantile([0.02, 0.98])
    y_low, y_high = plot_df[y].quantile([0.02, 0.98])

    x_pad = (x_high - x_low) * 0.08 if x_high > x_low else 0.5
    y_pad = (y_high - y_low) * 0.08 if y_high > y_low else 0.5

    return (x_low - x_pad, x_high + x_pad), (y_low - y_pad, y_high + y_pad)


def plot_one_condition(
    df: pd.DataFrame,
    x: str,
    y: str,
    condition: str,
    filename: str,
) -> None:
    plot_df = df[df["condition"] == condition].copy()

    if plot_df.empty:
        print(f"Skipping {filename}: no data for {condition}")
        return

    xlim, ylim = get_local_limits(plot_df, x, y)

    plt.figure(figsize=(7, 6))

    sns.scatterplot(
        data=plot_df,
        x=x,
        y=y,
        alpha=0.3
    )

    sns.regplot(
        data=plot_df,
        x=x,
        y=y,
        scatter=False
    )

    plt.xlim(xlim)
    plt.ylim(ylim)

    plt.xlabel(x.replace("_", " "))
    plt.ylabel(y.replace("_", " "))
    plt.title(f"{condition}: {y} vs {x}")
    plt.tight_layout()

    out_path = OUTPUT_DIR / filename
    plt.savefig(out_path, dpi=300)
    plt.close()

    print("Saved:", out_path)


def main():
    print("ROOT:", ROOT)
    print("INPUT_DIR:", INPUT_DIR)
    print("OUTPUT_DIR:", OUTPUT_DIR)

    # TEMPERATURA
    df_temp = build_pair_df(
        "t_microclimateback.csv", "t_microclimateback",
        "t_dorsal.csv", "t_dorsal"
    )
    print_pair_debug(df_temp, "t_microclimateback", "t_dorsal")

    for condition in CONDITION_ORDER:
        plot_one_condition(
            df_temp,
            x="t_microclimateback",
            y="t_dorsal",
            condition=condition,
            filename=f"{condition}_t_microclimateback_vs_t_dorsal.png",
        )

    # UMIDITA
    df_rh = build_pair_df(
        "rh_microclimateback.csv", "rh_microclimateback",
        "rh_dorsal.csv", "rh_dorsal"
    )
    print_pair_debug(df_rh, "rh_microclimateback", "rh_dorsal")

    for condition in CONDITION_ORDER:
        plot_one_condition(
            df_rh,
            x="rh_microclimateback",
            y="rh_dorsal",
            condition=condition,
            filename=f"{condition}_rh_microclimateback_vs_rh_dorsal.png",
        )

    print(f"\nDone. Mechanistic plots saved in:\n{OUTPUT_DIR}")


if __name__ == "__main__":
    main()

ROOT: /home/eleonorab/repository/convective_cooling
INPUT_DIR: /home/eleonorab/repository/convective_cooling/data/processed/aggregated_by_signal
OUTPUT_DIR: /home/eleonorab/repository/convective_cooling/reports/mechanistic_plots

PAIR DATASET: t_dorsal vs t_microclimateback
shape: (1078, 6)
conditions: {'no_wind': 360, 'wind_low': 359, 'wind_high': 359}

DEBUG for t_dorsal vs t_microclimateback

CONDITION: no_wind
rows: 360
t_microclimateback non-null: 360
t_dorsal non-null: 360

CONDITION: wind_low
rows: 359
t_microclimateback non-null: 359
t_dorsal non-null: 359

CONDITION: wind_high
rows: 359
t_microclimateback non-null: 359
t_dorsal non-null: 359
Saved: /home/eleonorab/repository/convective_cooling/reports/mechanistic_plots/no_wind_t_microclimateback_vs_t_dorsal.png
Saved: /home/eleonorab/repository/convective_cooling/reports/mechanistic_plots/wind_low_t_microclimateback_vs_t_dorsal.png
Saved: /home/eleonorab/repository/convective_cooling/reports/mechanistic_plots/wind_high_t_micro